In [60]:
import os
import json
from openai import OpenAI
from typing import List, Dict
import csv
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.core import Feedback
from dotenv import load_dotenv
from trulens.apps.custom import instrument

load_dotenv()


True

In [61]:
with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 
    
len(queries)

23

In [64]:


session = TruSession()

## Uncomment the following to reset database 
# session.reset_database()

In [65]:
llm = OpenAI()

In [66]:
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [67]:
embed = llm.embeddings.create


In [68]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            k=5
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=k,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            
            return docs

       


In [69]:
openai_retreiver = retriever(embed, index)

In [70]:
system_prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"


In [71]:
class generator:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"""Please generate a comprehensive response to this query:

Original Query: {query}

Using the following information:

Initial Context:
{formatted_context}"""
        }
    ]
)
        evaluation_response = json.loads(self._evaluate(query,context,response.choices[0].message.content))
        regeneration, reretrieval,query_new = evaluation_response["requires_regeneration"],evaluation_response["requires_retrieval"],evaluation_response["query"]
        print(f"reretrieving: {reretrieval}, regeneration: {regeneration}")
        if reretrieval or regeneration:
            print(query_new)
            context_new = self.retriever.get_data(query =query_new)
            context_new.extend(context) 
            formatted_context_new = "\n".join([str(doc) for doc in context_new])
            response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"""Please generate a comprehensive response to this query:

Original Query: {query_new}

Using the following information:

Initial Context:
{formatted_context_new}"""
        }
    ]
)
        return response.choices[0].message.content
    


    def _evaluate(self,query, context, generation):
        schema= {
    "type": "json_schema",
    "json_schema": {
      "name": "llm_evaluation",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": {
          "requires_retrieval": {
            "type": "boolean",
            "description": "Indicates if additional context retrieval is needed."
          },
          "query": {
            "type": "string",
            "description": "A query to get extra context chunks for missing information."
          },
          "requires_regeneration": {
            "type": "boolean",
            "description": "Indicates if the response needs to be regenerated."
          }
        },
        "required": [
          "requires_retrieval",
          "query",
          "requires_regeneration"
        ],
        "additionalProperties": False
      }
    }
  }
        response = llm.chat.completions.create(
  model="gpt-4o",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "You are an AI evaluator responsible for determining the quality of a generated response\nin a Retrieval-Augmented Generation (RAG) system. Your role is to evaluate the\naccuracy, relevance, and completeness of the response based on the provided query and the context chunks.\n\n### Task:\nEvaluate whether the generated response sufficiently answers the query using the retrieved context\nand determine if:\n1. Additional retrieval is required to provide a better answer.\n2. The response should be regenerated based on its quality.3.Check for questions for entities or concepts that are not explicitly mentioned for example to answer what are the health benefits of the machine that burns most claories during cardio? if the chunks does not mention the machine that burns most calories you need to perform a query like (what is the cardio machine that burns most calories and add context to the first question to provide good answer\n\n### Input:\n1. **Query**: The user's input or question.\n2. **Context Chunks**: A list of text chunks retrieved from a knowledge base, intended to assist in answering the query.\n3. **Generated Response**: The system's response to the query.\n\n### Output:\nProvide your evaluation as a structured JSON object in the following format:\n- \"query\": ask a question to clarify any missing information or to provide more context that is misisng from the chunks to answer the query asked. you can also ask a query to provide context that is missing\n- \"retrieved_context\": A summary of the retrieved context chunks or the key points.\n- \"generated_response\": Restate the generated response.\n- \"verdict\":\n  - \"_requires_retrieval\": A boolean indicating whether additional context retrieval is necessary.\n  - \"_retrieval_query\": If additional retrieval is needed, provide a new query or refinement to guide the retrieval process. Otherwise, this field should be null.\n  - \"_requires_regeneration\": A boolean indicating whether the generated response needs to be regenerated.\n- \"overall_feedback\": A concise explanation of your reasoning, including what is missing or why the response is sufficient.\n\n### Example Input:\nQuery: \"What are the health benefits of regular exercise for the machine that bruns most caloris for cardio?\"\nContext Chunks:\n1. \"Regular exercise improves cardiovascular health by strengthening the heart and improving blood circulation.\"\n2. \"It helps with weight management by burning calories and building muscle.\"\n3. \"Exercise reduces stress and anxiety by releasing endorphins.\"\n4. \"It also improves sleep quality and increases energy levels.\"\nGenerated Response: \"Regular exercise strengthens the heart and helps manage weight.\"\n\n### Example Output:\n{\n    \"query\": \"what is the cardio machine the burns most calories?\",\n    \"retrieved_context\": [\n        \"Regular exercise improves cardiovascular health by strengthening the heart and improving blood circulation.\",\n        \"It helps with weight management by burning calories and building muscle.\",\n        \"Exercise reduces stress and anxiety by releasing endorphins.\",\n        \"It also improves sleep quality and increases energy levels.\"\n    ],\n    \"generated_response\": \"Regular exercise strengthens the heart and helps manage weight.\",\n    \"verdict\": {\n        \"_requires_retrieval\": True,\n        \"_retrieval_query\": \"Provide additional information on the mental health and sleep benefits of regular exercise.\",\n        \"_requires_regeneration\": True\n    },\n    \"overall_feedback\": \"The response is incomplete as it omits critical details about reducing stress, improving sleep quality, and increasing energy levels. Additional retrieval focused on mental health and sleep benefits is recommended. The context provided so far is insufficient, and the response needs regeneration based on more comprehensive retrieval.\"\n}\n\n### Instructions:\nEvaluate the provided query, context chunks, and generated response. If additional retrieval is required, specify a refined query for retrieval in the verdict object. Provide your output in the same structured JSON format as shown above."
        }
      ],
    },
    {
       "role": "user",
      "content": [
          {    
          "type":"text",
          "text": f"query: {query}\nresponse:{generation}\nchunks {context} "
          }
      ] 
    }
  ],
  response_format=schema,
  temperature=0,
  max_completion_tokens=250,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
)
        return response.choices[0].message.content



In [72]:
openai_generator = generator(llm,openai_retreiver)

In [ ]:
# ## Test
# context = openai_retreiver.get_data(queries[21])
# eval = openai_generator.generate(queries[21], context)

reretrieving: False, regeneration: False


In [58]:
eval

"For medical professionals over 60 years old who wish to continue their practice, there are specific documentation requirements that need to be fulfilled. These requirements include providing a recent medical report along with other general documents such as a valid insurance against medical faults, a valid labor card, a valid national identity card, and a valid visa copy. In addition to these, a recent Continuous Medical Education (CME) certificate, which must account for 40 hours, is necessary. If the professional is 60 years or older, a medical fitness certificate is also required. The application for license renewal involves an application fee of 100 AED, and for doctors, the renewal fee is 3,000 AED in the private sector; however, it is free in the government sector. The process can be completed instantly via service channels like the MOHAP website. A letter from the facility requesting the re-licensing, the doctor's contract of employment, a copy of the valid license, an assessme

In [74]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response

In [75]:
ret = retriever(embed, index)
gen= generator(llm,ret)

In [40]:
rag_app = Rag_app(gen, ret)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [76]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="SELF RAG",
    app_version="4o-large_3-500-01",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [77]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [80]:
with tru_rag as recording:
    for eval in queries[0:17]:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

How do I register for controlled or semi-controlled drugs custody?
reretrieving: False, regeneration: False
What are the requirements for renewing the registration of a conventional pharmaceutical product?
reretrieving: False, regeneration: False
How do I appeal a decision made by the Medical Licensing Committee?
reretrieving: False, regeneration: False
What is the process for obtaining a certificate of amendment for registered pharmaceutical products?
reretrieving: False, regeneration: False
How can I get a product classified?
reretrieving: False, regeneration: False
What are the steps to re-license a pharmaceutical facility?
reretrieving: False, regeneration: False
How can I renew my license as a nurse or medical professional?
reretrieving: False, regeneration: False
What's the process for getting a permit to import medical equipment?
reretrieving: False, regeneration: False
How can I renew my health facility license?
reretrieving: False, regeneration: False
What are the steps to reg

In [22]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at http://192.168.1.12:56740 .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

In [ ]:
# ## Test
# query="what is the the service i use to register as new practicing doctor in the UAE "
# res = openai_retreiver.get_data(query)
# print(res)
# print(type(res))